In [2]:
#Paso 1. Cargar el conjunto de datos

import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/4GeeksAcademy/NLP-project-tutorial/main/url_spam.csv")

df.head()

,url,is_spam
0,https://briefingday.us8.list-manage.com/unsubs...,True
1,https://www.hvper.com/,True
2,https://briefingday.com/m/v4n3i4f3,True
3,https://briefingday.com/n/20200618/m#commentform,False
4,https://briefingday.com/fan,True


Paso 2. Inspeccionar los datos
Comprobamos las dimensiones, los nombres de las columnas y los valores ausentes para comprender los datos antes de procesarlos.

In [3]:
print(df.shape)
print(df.columns)
print(df.isnull().sum())

(2999, 2)
Index(['url', 'is_spam'], dtype='str')
url        0
is_spam    0
dtype: int64


In [5]:
# Importar las herramientas de preprocesamiento

import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [6]:
#Importar las herramientas de preprocesamiento
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_url(url):
    url = str(url).lower()
    tokens = re.split(r"[^a-zA-Z0-9]+", url)

    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token and token not in stop_words
    ]

    return " ".join(tokens)

In [8]:
#comprobamos los nombres de la columna 
df.columns

Index(['url', 'is_spam'], dtype='str')

In [9]:
df["url_processed"] = df["url"].apply(preprocess_url)

df[["url", "url_processed"]].head()

,url,url_processed
0,https://briefingday.us8.list-manage.com/unsubs...,http briefingday us8 list manage com unsubscribe
1,https://www.hvper.com/,http www hvper com
2,https://briefingday.com/m/v4n3i4f3,http briefingday com v4n3i4f3
3,https://briefingday.com/n/20200618/m#commentform,http briefingday com n 20200618 commentform
4,https://briefingday.com/fan,http briefingday com fan


Entrenamiento del modelo SVM

Dividimos el conjunto de datos en entrenamiento y prueba. Utilizamos TF-IDF para convertir las URLs en vectores numéricos y entrenamos un modelo SVM con sus parámetros por defecto.

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

X = df["url_processed"]
y = df["is_spam"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("svm", SVC())
])

 Optimización de hiperparámetros

Utilizamos GridSearchCV para probar diferentes configuraciones del SVM y seleccionar los hiperparámetros que obtienen mejores resultados mediante validación cruzada.

In [12]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "svm__C": [0.1, 1, 10],
    "svm__kernel": ["linear", "rbf"]
}

grid_search = GridSearchCV(
    svm_pipeline,
    param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Mejores parámetros:", grid_search.best_params_)
print("Mejor resultado:", grid_search.best_score_)

Mejores parámetros: {'svm__C': 1, 'svm__kernel': 'rbf', 'tfidf__ngram_range': (1, 2)}
Mejor resultado: 0.908378114812496


Guardado del modelo

Guardamos el mejor modelo entrenado utilizando Joblib para poder reutilizarlo posteriormente sin necesidad de volver a entrenarlo.

In [13]:
import joblib
from pathlib import Path

Path("models").mkdir(exist_ok=True)

joblib.dump(
    grid_search.best_estimator_,
    "models/url_spam_svm.joblib"
)

['models/url_spam_svm.joblib']